# Сравнение. Предсказание элемента с помощью RNN и с помощью марковских цепей


## Задание

Сравните работу программы предсказания с помощью RNN и с помощью Марковских цепей (установите одинаковые параметры). Что лучше (возвращаемся к вопросу о критериях качества). ( для сравнения на уровне букв настройте и используйте файл Corpus.py -  без GUI)

---

In [ ]:
import numpy as np
import re
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from tensorflow.keras.layers import Dense, SimpleRNN, Input, Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

import matplotlib.pyplot as plt
from collections import defaultdict, Counter
import random

In [ ]:
class MarkovChain:
    def __init__(self, order=3):
        self.order = order
        self.chain = defaultdict(Counter)
        self.start_tokens = []
        
    def fit(self, text):
        """Обучение марковской цепи на тексте"""
        words = text.split()
        
        if len(words) < self.order:
            raise ValueError(f"Текст слишком короткий для порядка {self.order}")
            
        # Сохраняем начальные последовательности
        for i in range(len(words) - self.order + 1):
            sequence = tuple(words[i:i + self.order])
            if i == 0:
                self.start_tokens.append(sequence)
                
            if i + self.order < len(words):
                next_word = words[i + self.order]
                self.chain[sequence][next_word] += 1
                
    def generate(self, length=50, start_sequence=None):
        """Генерация текста заданной длины"""
        if not self.chain:
            raise ValueError("Модель не обучена")
            
        if start_sequence:
            current = tuple(start_sequence.split()[:self.order])
            if len(current) < self.order:
                current = random.choice(self.start_tokens)
        else:
            current = random.choice(self.start_tokens)
            
        result = list(current)
        
        for _ in range(length - self.order):
            if current not in self.chain:
                break
                
            next_words = list(self.chain[current].keys())
            weights = list(self.chain[current].values())
            
            next_word = random.choices(next_words, weights=weights)[0]
            result.append(next_word)
            
            current = tuple(result[-self.order:])
            
        return ' '.join(result)

class MarkovCharChain:
    """Марковская цепь для работы с символами"""
    def __init__(self, order=3):
        self.order = order
        self.chain = defaultdict(Counter)
        self.start_sequences = []
        
    def fit(self, text):
        """Обучение на символьном уровне"""
        if len(text) < self.order:
            raise ValueError(f"Текст слишком короткий для порядка {self.order}")
            
        for i in range(len(text) - self.order):
            sequence = text[i:i + self.order]
            if i == 0:
                self.start_sequences.append(sequence)
                
            next_char = text[i + self.order]
            self.chain[sequence][next_char] += 1
                
    def generate(self, length=100, start_sequence=None):
        """Генерация текста на символьном уровне"""
        if not self.chain:
            raise ValueError("Модель не обучена")
            
        if start_sequence and len(start_sequence) >= self.order:
            current = start_sequence[:self.order]
        else:
            current = random.choice(self.start_sequences)
            
        result = current
        
        for _ in range(length - self.order):
            if current not in self.chain:
                break
                
            next_chars = list(self.chain[current].keys())
            weights = list(self.chain[current].values())
            
            next_char = random.choices(next_chars, weights=weights)[0]
            result += next_char
            current = result[-self.order:]
            
        return result

In [ ]:
def load_and_preprocess_text(filename, char_level=False):
    """Загрузка и предобработка текста"""
    with open(filename, 'r', encoding='utf-8') as f:
        text = f.read()
        text = text.replace('\ufeff', '')
        
        if char_level:
            text = re.sub(r'[^А-яёЁ ]', ' ', text)
            text = re.sub(r'\s+', ' ', text).strip().lower()
        else:
            text = re.sub(r'\s+', ' ', text).strip().lower()
            
        return text

text = load_and_preprocess_text('cat.txt', char_level=False)
char_text = load_and_preprocess_text('cat.txt', char_level=True)
print(f"Длина текста (слова): {len(text)} символов")
print(f"Длина текста (символы): {len(char_text)} символов")
print(f"Первые 500 символов текста:\n{text[:500]}...")

In [ ]:
def evaluate_coherence(text, generated_text):
    """Оценка когерентности сгенерированного текста"""
    # Простые метрики для оценки качества
    original_words = text.split()
    generated_words = generated_text.split()
    
    # 1. Доля уникальных слов (меньше - лучше для когерентности)
    unique_ratio = len(set(generated_words)) / len(generated_words) if generated_words else 1
    
    # 2. Средняя длина слова
    avg_word_length = np.mean([len(word) for word in generated_words]) if generated_words else 0
    
    # 3. Доля реальных русских слов (простая проверка)
    russian_chars = set('абвгдеёжзийклмнопрстуфхцчшщъыьэюя')
    real_word_ratio = sum(1 for word in generated_words 
                         if len(word) > 1 and all(c in russian_chars for c in word)) / len(generated_words) if generated_words else 0
    
    return {
        'unique_ratio': unique_ratio,
        'avg_word_length': avg_word_length,
        'real_word_ratio': real_word_ratio,
        'length': len(generated_text)
    }

# Обучаем модели на уровне символов
print("=== ЭКСПЕРИМЕНТ 1: СРАВНЕНИЕ НА УРОВНЕ СИМВОЛОВ ===")

# Марковская цепь для символов
markov_char = MarkovCharChain(order=3)
markov_char.fit(char_text[:2000])  # Обучаем на первых 2000 символах

# RNN для символов
num_characters = 34  # 33 буквы + пробел
tokenizer_char = Tokenizer(num_words=num_characters, char_level=True)
tokenizer_char.fit_on_texts(char_text)

inp_chars = 3
data = tokenizer_char.texts_to_matrix(char_text[:2000])
n = data.shape[0] - inp_chars

if n > 0:
    X = np.array([data[i:i + inp_chars, :] for i in range(n)])
    Y = data[inp_chars:]

    # Создаем RNN модель
    rnn_char_model = Sequential([
        Input((inp_chars, num_characters)),
        SimpleRNN(128, activation='tanh'),
        Dense(num_characters, activation='softmax')
    ])
    
    rnn_char_model.compile(loss='categorical_crossentropy', 
                          metrics=['accuracy'], 
                          optimizer='adam')
    
    print("Обучение RNN на символах...")
    rnn_char_history = rnn_char_model.fit(X, Y, batch_size=32, epochs=20, verbose=False)
    
    # Функция для генерации текста RNN
    def rnn_char_generate(start_text, length=100):
        result = start_text
        for i in range(length):
            x = []
            current_seq = result[-inp_chars:] if len(result) >= inp_chars else result
            current_seq = current_seq.ljust(inp_chars, ' ')[-inp_chars:]
            
            for char in current_seq:
                x.append(tokenizer_char.texts_to_matrix(char))
            
            x = np.array(x)
            inp = x.reshape(1, inp_chars, num_characters)
            
            pred = rnn_char_model.predict(inp, verbose=False)
            next_char_idx = pred.argmax(axis=1)[0]
            
            if next_char_idx in tokenizer_char.index_word:
                next_char = tokenizer_char.index_word[next_char_idx]
                result += next_char
            else:
                break
                
        return result

    # Генерируем тексты
    start_sequences = ["кот", "лес", "дуб"]
    
    print("\n--- ГЕНЕРАЦИЯ ТЕКСТА ---")
    for start_seq in start_sequences:
        print(f"\nСтартовая последовательность: '{start_seq}'")
        
        # Марковская цепь
        markov_result = markov_char.generate(length=100, start_sequence=start_seq)
        print(f"Марковская цепь: {markov_result}")
        
        # RNN
        rnn_result = rnn_char_generate(start_seq, length=100)
        print(f"RNN: {rnn_result}")
        
        # Оценка качества
        markov_metrics = evaluate_coherence(char_text, markov_result)
        rnn_metrics = evaluate_coherence(char_text, rnn_result)
        
        print(f"Марков - Уникальность: {markov_metrics['unique_ratio']:.3f}, "
              f"Реальные слова: {markov_metrics['real_word_ratio']:.3f}")
        print(f"RNN - Уникальность: {rnn_metrics['unique_ratio']:.3f}, "
              f"Реальные слова: {rnn_metrics['real_word_ratio']:.3f}")

Марков не генерирует с некоторых стартовых последовательностей использует разнообразные слова (псевдослова), которые не вызывают ощущения белеберды (коей являются) 

RNN - зациклилось на одну и ту же последовательность, взяв реальное слово сто и "ве" - может вдохновился словом не, но что-то пошло не так

In [ ]:
print("\n=== ЭКСПЕРИМЕНТ 2: СРАВНЕНИЕ НА УРОВНЕ СЛОВ ===")

# Подготовка текста для словесного уровня
words = text.split()
vocab_size = min(1000, len(set(words)))

# Марковская цепь для слов
markov_word = MarkovChain(order=2)
markov_word.fit(text)

# RNN для слов
tokenizer_word = Tokenizer(num_words=vocab_size, filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\\t\\n',
                          lower=True, split=' ', char_level=False)
tokenizer_word.fit_on_texts([text])

data_words = tokenizer_word.texts_to_sequences([text])[0]
data_words = np.array(data_words)

inp_words = 2
n_words = data_words.shape[0] - inp_words

if n_words > 0:
    X_words = np.array([data_words[i:i + inp_words] for i in range(n_words)])
    Y_words = to_categorical(data_words[inp_words:], num_classes=vocab_size)

    # RNN модель для слов
    rnn_word_model = Sequential([
        Embedding(vocab_size, 50, input_length=inp_words),
        SimpleRNN(100, activation='tanh'),
        Dense(vocab_size, activation='softmax')
    ])
    
    rnn_word_model.compile(loss='categorical_crossentropy', 
                          metrics=['accuracy'], 
                          optimizer='adam')
    
    print("Обучение RNN на словах...")
    rnn_word_history = rnn_word_model.fit(X_words, Y_words, batch_size=32, epochs=30, verbose=False)
    
    # Функция генерации для RNN слов
    def rnn_word_generate(start_text, length=20):
        result = start_text.split()
        data_seq = tokenizer_word.texts_to_sequences([start_text])[0]
        
        for i in range(length):
            if len(data_seq) < inp_words:
                current_seq = data_seq
                current_seq = [0] * (inp_words - len(current_seq)) + current_seq
            else:
                current_seq = data_seq[-inp_words:]
            
            inp = np.expand_dims(current_seq, axis=0)
            pred = rnn_word_model.predict(inp, verbose=False)
            next_word_idx = pred.argmax(axis=1)[0]
            
            if next_word_idx in tokenizer_word.index_word and next_word_idx != 0:
                next_word = tokenizer_word.index_word[next_word_idx]
                result.append(next_word)
                data_seq.append(next_word_idx)
            else:
                break
                
        return ' '.join(result)

    # Генерация и сравнение
    start_phrases = ["кот сидел", "старый дуб", "в лесу"]
    
    print("\n--- ГЕНЕРАЦИЯ ТЕКСТА (СЛОВА) ---")
    for start_phrase in start_phrases:
        print(f"\nСтартовая фраза: '{start_phrase}'")
        
        # Марковская цепь
        markov_word_result = markov_word.generate(length=15, start_sequence=start_phrase)
        print(f"Марковская цепь: {markov_word_result}")
        
        # RNN
        rnn_word_result = rnn_word_generate(start_phrase, length=15)
        print(f"RNN: {rnn_word_result}")
        
        # Оценка качества
        markov_word_metrics = evaluate_coherence(text, markov_word_result)
        rnn_word_metrics = evaluate_coherence(text, rnn_word_result)
        
        print(f"Марков - Уникальность: {markov_word_metrics['unique_ratio']:.3f}, "
              f"Длина: {markov_word_metrics['length']}")
        print(f"RNN - Уникальность: {rnn_word_metrics['unique_ratio']:.3f}, "
              f"Длина: {rnn_word_metrics['length']}")

Марковская цепь не сгенерировала НИЧЕГО
У RNN разнообразные результаты:
1) очень похожий (в пределах нашей работы) текст
2) нашёл достаточно длинную цепочку и зациклился
3) зациклился на очень короткой и бессмысленной цепочке

In [ ]:
def calculate_perplexity(text, model_type='markov', order=3):
    """Вычисление перплексии (метрика качества языковых моделей)"""
    words = text.split()
    if len(words) < order + 1:
        return float('inf')
    
    log_prob = 0
    count = 0
    
    for i in range(len(words) - order):
        sequence = tuple(words[i:i + order])
        next_word = words[i + order]
        
        if model_type == 'markov':
            if sequence in markov_word.chain:
                total = sum(markov_word.chain[sequence].values())
                prob = markov_word.chain[sequence].get(next_word, 0) / total
            else:
                prob = 1e-6
        else:
            prob = 1e-5
            
        log_prob += np.log(prob + 1e-10)
        count += 1
    
    if count == 0:
        return float('inf')
        
    avg_log_prob = log_prob / count
    perplexity = np.exp(-avg_log_prob)
    
    return perplexity

print("\n=== СРАВНИТЕЛЬНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ ===")

results = []

test_sequences = ["кот", "лес", "дуб", "старый", "мудрый"]

for seq in test_sequences:
    markov_text = markov_char.generate(length=50, start_sequence=seq)
    rnn_text = rnn_char_generate(seq, length=50) if 'rnn_char_generate' in locals() else "N/A"
    
    markov_metrics = evaluate_coherence(char_text, markov_text)
    
    if rnn_text != "N/A":
        rnn_metrics = evaluate_coherence(char_text, rnn_text)
    else:
        rnn_metrics = {'unique_ratio': 0, 'real_word_ratio': 0, 'length': 0}
    
    results.append({
        'sequence': seq,
        'markov_unique': markov_metrics['unique_ratio'],
        'markov_real_words': markov_metrics['real_word_ratio'],
        'rnn_unique': rnn_metrics['unique_ratio'],
        'rnn_real_words': rnn_metrics['real_word_ratio'],
        'markov_text': markov_text[:50] + "...",
        'rnn_text': rnn_text[:50] + "..." if rnn_text != "N/A" else "N/A"
    })

print("\nРезультаты сравнения:")
print("Старт\tМарков-уник\tМарков-реаль\tRNN-уник\tRNN-реаль")
for res in results:
    print(f"{res['sequence']}\t{res['markov_unique']:.3f}\t\t{res['markov_real_words']:.3f}\t\t"
          f"{res['rnn_unique']:.3f}\t\t{res['rnn_real_words']:.3f}")

print("\n--- ПРИМЕРЫ СГЕНЕРИРОВАННЫХ ТЕКСТОВ ---")
for res in results:
    print(f"\nСтарт: '{res['sequence']}'")
    print(f"Марков: {res['markov_text']}")
    print(f"RNN: {res['rnn_text']}")

## Цифровые показатели

**Марковские цепи:**
- Уникальность: 100% по всем тестам
- Реальные слова: 87.5-100% (в среднем 95%)

**RNN:**
- Уникальность: 21-50% (очень низкая)
- Реальные слова: 80-100% (в среднем 91%)

## Качество генерации

**Марковские цепи:**
- Отличное разнообразие, но нестабильное качество
- Для "лес", "дуб", "мудрый" - генерирует только стартовое слово
- Для "кот", "старый" - создает текст с псевдословами ("застличных", "статов", "больствои")

**RNN:**
- Сильное зацикливание на паттернах:
  - "кот" → "сто ве" повторяется
  - "лес" → "лесть сто ве" повторяется  
  - "дуб" → "дубеть сто ве" повторяется
  - "старый", "мудрый" → "значит порешь и прой" повторяется
- Все слова реальные, но сочетания бессмысленные

## Вывод
Оба метода показали неудовлетворительные результаты:
- Марков либо молчит, либо генерирует бессмыслицу
- RNN зациклился на абсурдных повторениях
- Ни один метод не смог генерировать даже около осмысленный текст